In [ ]:
import pymysql
import pandas as pd

# 建立连接（你可以替换自己的连接信息）
conn = pymysql.connect(
    host='###',
    port=3306,
    user='###',
    password='###',
    database='ch00',
    charset='utf8'
)
query_full = """
SELECT dt, ticker, share, close, last_hold
FROM ppLog
WHERE boid = 43
  AND dt BETWEEN '2024-03-02' AND '2025-12-31'
ORDER BY dt, ticker
"""
df_full = pd.read_sql(query_full, conn)
print(df_full)

In [ ]:
# 按 ticker 排序后，对每只股票计算价格变动
df_full = df_full.sort_values(['ticker', 'dt']).copy()

df_full['close_shift'] = df_full.groupby('ticker')['close'].shift(1)
df_full['price_change'] = df_full['close'] - df_full['close_shift']

df_full['return'] = df_full['price_change'] / df_full['close_shift']
df_long = df_full[df_full['last_hold'] > 0].copy()
df_long['value'] = df_long['last_hold'] * df_long['close']

# 总市值
total_value = df_long.groupby('dt')['value'].sum().rename('total_value')
df_long = df_long.merge(total_value, on='dt')

df_long['weight'] = df_long['value'] / df_long['total_value']
df_long['weighted_return'] = df_long['weight'] * df_long['return']
portfolio_returns = df_long.groupby('dt')['weighted_return'].sum().sort_index()
print(df_long)

In [ ]:
def turnover(prev_w: pd.Series, curr_w: pd.Series) -> float:
    # 统一索引为两天权重的并集
    idx = prev_w.index.union(curr_w.index)
    pw = prev_w.reindex(idx, fill_value=0.0)
    cw = curr_w.reindex(idx, fill_value=0.0)
    return 0.5 * (cw - pw).abs().sum()


In [ ]:
turnover_list = []

# 获取所有交易日（已排序）
all_days = df_long['dt'].sort_values().unique()

for i in range(1, len(all_days)):
    prev_day = all_days[i-1]
    curr_day = all_days[i]

    # 前一天的权重（Series: index=ticker, value=weight）
    prev_w = df_long.loc[df_long['dt'] == prev_day].set_index('ticker')['weight']
    # 当天的权重
    curr_w = df_long.loc[df_long['dt'] == curr_day].set_index('ticker')['weight']

    # 用你定义好的 turnover 函数
    to = turnover(prev_w, curr_w)
    turnover_list.append((curr_day, to))
    print(to)

# 转换成 DataFrame
df_turnover = pd.DataFrame(turnover_list, columns=['dt', 'turnover'])
print(df_turnover)



In [ ]:
import matplotlib.pyplot as plt

cumulative_return = (1 + portfolio_returns).cumprod()-1

plt.figure(figsize=(14, 6))
plt.plot(cumulative_return.index, cumulative_return.values, label='return', marker='o')

plt.title("return curve")
plt.xlabel("date")
plt.ylabel("return")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 示例 SQL 查询（你可以替换为你的实际数据库操作）
query = """
SELECT dt, last, diff
FROM rt
WHERE ticker = '000300.SH' and dt>'20240301'
ORDER BY dt ASC
"""

# 读取数据
df_index = pd.read_sql(query, conn)  # conn 是你的数据库连接
print(df_index)

In [ ]:
df_index = df_index.sort_values('dt').copy()  # 按日期排序
df_index['last_shift'] = df_index['last'].shift(1)  # 前一日的 last

# 计算每日收益率（注意第一天结果为 NaN）
df_index['index_return'] = df_index['diff'] / df_index['last_shift']


df_index['dt'] = pd.to_datetime(df_index['dt'])  # 转换为 datetime 类型
df_index['dt'] = df_index['dt'].dt.normalize()  # 保留日期，丢掉时间部分，但仍是 datetime 类型

df_index = df_index.set_index('dt')  # 正确设置 index

# 计算累计收益率（从第二天开始）
df_index['index_cum_return'] = (1 + df_index['index_return']).cumprod()-1

print(df_index)
print(df_index.index)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(cumulative_return.index, cumulative_return.values, label='portfolio return', marker='o')
plt.plot(df_index.index, df_index['index_cum_return'], marker='s', label='index return')

plt.title('中证300指数累计收益率曲线')
plt.xlabel('日期')
plt.ylabel('累计收益率')
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
conn = pymysql.connect(host='###', user='###', password='###', db='ch00')

query = """
SELECT DISTINCT ticker
FROM pplog
WHERE boid = 43
  AND dt BETWEEN '2023-08-29' AND '2025-12-31';
"""

df_candidate_tickers = pd.read_sql(query, conn)
tickers = df_candidate_tickers['ticker'].unique().tolist()
print(len(tickers))


# 构建 ticker 列表字符串
tickers_str = ','.join(f"'{t}'" for t in tickers)
print(tickers_str)

query = f"""
SELECT locdt, ticker, last, diff
FROM rt
WHERE ticker IN ({tickers_str})
  AND locdt BETWEEN '2023-08-29' AND '2025-12-31';
"""

df_price = pd.read_sql(query, conn)

# 重命名列
df_price = df_price.rename(columns={
    'locdt': 'dt',
    'last': 'close',
    'diff': 'diff'  # 保持 diff 不变
})

# 转换为日期格式（如需要）
df_price['dt'] = pd.to_datetime(df_price['dt'])

# 可选：按日期排序，检查数据
df_price = df_price.sort_values(['dt', 'ticker'])
print(df_price.head())
print(df_price.shape)

In [ ]:
query = """
SELECT dt, ticker, age
FROM pplog
WHERE boid = 43
  AND dt BETWEEN '2023-08-29' AND '2025-12-31';
"""

df_holdings = pd.read_sql(query, conn)
df_holdings['dt'] = pd.to_datetime(df_holdings['dt'])

# 检查前几天的持仓情况
print(df_holdings.groupby('dt').ticker.apply(list).head())

In [ ]:
import cvxpy as cp
import numpy as np
def optimization_from_mu_sigma(mu, Sigma,w_prev=None, turnover_cap=None,Base_today=None):
    n = len(mu)
    w = cp.Variable(n)
    
    expected_return = mu @ w
    risk_penalty = cp.quad_form(w, Sigma)
    objective = cp.Maximize(expected_return)

    constraints = [
        cp.sum(w) == 1,
        cp.quad_form(w, Sigma) <= 0.005,
        w >= 0,
    ]

    per_asset_cap=0.02
    # 总换手上限（0.5 * L1 口径）
    '''if w_prev is not None:
        # 只新增这几行：按当前 tickers 对齐并补0
        if isinstance(w_prev, pd.Series):
            prev = w_prev.reindex(tickers, fill_value=0.0).values
        else:
            prev = np.asarray(w_prev)
            if prev.size!=len(tickers):
                raise ValueError(111)
        constraints+=[w-prev<=per_asset_cap,prev-w<=per_asset_cap]'''

    if Base_today is not None and not np.allclose(Base_today, 0):
        if isinstance(Base_today, pd.Series):
            Base_today = Base_today.reindex(tickers, fill_value=0.0).values
        else:
            Base_today = np.asarray(Base_today)
        constraints += [w - Base_today <= 0.03,
                        w - Base_today >= -0.03]
        
    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.MOSEK)

    if w.value is not None:
        return pd.Series(w.value, index=tickers_age1)
    else:
        print("⚠️ 优化失败")
        return pd.Series([np.nan] * n, index=tickers_age1)

In [ ]:
def clip_daily_percentile(df, lower_pct, upper_pct):
    """
    对 DataFrame 按行（每天）进行 clip：每一行保留中间百分比的值，将极端值限制在分位数范围内。
    """
    q_low = df.quantile(lower_pct, axis=1)
    q_high = df.quantile(upper_pct, axis=1)

    # 对每一行执行 clip，用 apply 和 zip 逐行处理
    df_clipped = df.copy()

    for i in df.index:
        df_clipped.loc[i] = df.loc[i].clip(lower=q_low.loc[i], upper=q_high.loc[i])

    return df_clipped

In [ ]:
def turnover(prev_w: pd.Series, curr_w: pd.Series) -> float:
    # 统一索引为两天权重的并集
    idx = prev_w.index.union(curr_w.index)
    pw = prev_w.reindex(idx, fill_value=0.0)
    cw = curr_w.reindex(idx, fill_value=0.0)
    return 0.5 * (cw - pw).abs().sum()


def max_drawdown(series):
    series=series+1
    cum_max = series.cummax()             # 历史最高点
    drawdown = ( cum_max - series) / cum_max
    return drawdown.max()


In [ ]:
import warnings
warnings.filterwarnings("ignore")

df_price_wide = df_price.pivot(index='dt', columns='ticker', values='close')
all_days = df_price_wide.index.sort_values()  #所有日期参数
all_days = all_days[all_days != pd.Timestamp('2025-01-01')]

lookback_days_var = 120
holding_days = 1

target_days = all_days[lookback_days_var::holding_days]
opt_portfolio_returns=[]
rebalance_days = []
i=0
prev_weights = None
turnover_series=[]
df_long['dt'] = pd.to_datetime(df_long['dt']).dt.normalize()

for target_day in target_days:
    rebalance_days.append(target_day)
    tickers_today=df_holdings[df_holdings['dt']==target_day]['ticker'].unique().tolist()
        # 分两类
    tickers_age1 = df_holdings[(df_holdings['dt'] == target_day) & (df_holdings['age'] == 1)]['ticker'].unique().tolist()
    tickers_other = list(set(tickers_today) - set(tickers_age1))
    window_days = all_days[all_days<=target_day][-lookback_days_var:]
    df_window = df_price_wide.loc[window_days, tickers_today]
    returns_df = df_window.pct_change().dropna()

    sigma = returns_df.cov().values

    prev2_day = window_days[-3]
    prev1_day = window_days[-2]
    price_t_2 = df_price_wide.loc[prev2_day, tickers_today]
    price_t_1 = df_price_wide.loc[prev1_day, tickers_today]
    mu = (price_t_1 / price_t_2 - 1).values
    '''mu=np.clip(mu,-0.08,0.08)'''#如果需要缩尾
    tickers = returns_df.columns.tolist()

    target_day_norm = pd.to_datetime(target_day).normalize()
    base_today = (
        df_long.loc[df_long['dt'] == target_day_norm, ['ticker', 'weight']]
            .set_index('ticker')['weight']
            .reindex(tickers_today, fill_value=0.0)
    )

        ###
    if len(tickers_age1) > 0:
        returns_df_age1 = returns_df[tickers_age1]
        sigma_age1 = returns_df_age1.cov().values
        mu_age1 = mu[[tickers.index(t) for t in tickers_age1]]

        weights_age1 = optimization_from_mu_sigma(mu_age1, sigma_age1, prev_weights, base_today.loc[tickers_age1])
        weights_age1 = pd.Series(weights_age1, index=tickers_age1)

        # 缩放到总权重 20%
        weights_age1 = weights_age1 / weights_age1.sum() * 0.20

        # 先构造全体股票权重，非 age=1 的股票先填 0
        weights_all = pd.Series(0, index=tickers_today)
        weights_all.loc[tickers_age1] = weights_age1

        # 按 20% / 80% 合并
        weights = weights_all + 0.8 * base_today.reindex(tickers_today, fill_value=0)

    else:
        weights = base_today.reindex(tickers_today,fill_value=0)
        ###用于完成age=1部分


    '''weights = optimization_from_mu_sigma(mu,sigma,w_prev=prev_weights,Base_today=base_today) '''

    next_days = all_days[all_days >= target_day][:holding_days+1]
    if len(next_days) < holding_days:
        continue
    future_prices = df_price_wide.loc[next_days, tickers_today]
    future_returns = future_prices.pct_change().dropna()
    portfolio_ret = future_returns @ weights
    opt_portfolio_returns.append(portfolio_ret)

    if prev_weights is not None:
        # 权重变化的绝对值之和除以2
        change_ratio = turnover(prev_weights,weights)
        ##print(f"{target_day.date()} 调仓比例: {change_ratio:.4f}")
    else:
        # 第一天没有调仓比例
        change_ratio = np.nan
        
    if i==0:
        print(base_today.sort_values(ascending=False))
    if i ==3:
        print(base_today.sort_values(ascending=False))
        print(weights.sum(),23)
        print(weights.shape)
        print(prev_weights.shape)
    prev_weights = weights
    i+=1
    
opt_portfolio_returns=pd.concat(opt_portfolio_returns)


In [ ]:
cumulative_returns = (1 + opt_portfolio_returns).cumprod()-1

# 1) 统一索引类型
cumulative_returns.index = pd.to_datetime(cumulative_returns.index).normalize()
cumulative_return.index  = pd.to_datetime(cumulative_return.index).normalize()
df_index.index          = pd.to_datetime(df_index.index).normalize()

# 2) 设起始日为 Timestamp
start = pd.Timestamp('2025-01-01')

# 3) 过滤 2025 年之后（切片写法最省心）
cr_2025   = cumulative_returns.loc[start:]
ci_2025   = df_index.loc[start:]
ret_2025  = cumulative_return.loc[start:]


# 让三条曲线都从 0 开始
cr_2025  = cr_2025 - cr_2025.iloc[0]
ci_2025['index_cum_return'] = ci_2025['index_cum_return'] - ci_2025['index_cum_return'].iloc[0]
ret_2025 = ret_2025 - ret_2025.iloc[0]

# 4) 画图
plt.figure(figsize=(12, 6))
plt.plot(cr_2025.index,  cr_2025.values,  label='Optimized Portfolio', marker='o')
plt.plot(ci_2025.index,  ci_2025['index_cum_return'], label='index return', marker='s')
plt.plot(ret_2025.index, ret_2025.values, label='return', marker='o')

# 标注调仓日（只标 2025+ 的）
for day in rebalance_days:
    if day >= start:
        plt.axvline(x=day, color='red', linestyle='--', alpha=0.5)

plt.title('优化组合累计收益（2025+）'); plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()


# 总收益（区间最后一天的累计收益）
total_return_optimized = cr_2025.iloc[-1]
total_return_orginal= ret_2025.iloc[-1]

# 收益波动率（基于每日收益）
volatility_optimized = cr_2025.std()
volatility_orginal = ret_2025.std()

# 夏普比率（非年化）
sharpe_ratio_opt = (total_return_optimized-0.03) / volatility_optimized
sharpe_ratio_org = (total_return_orginal-0.03) / volatility_orginal
print("Sharpe Ratio (not annualized):", sharpe_ratio_opt)
print(sharpe_ratio_org,23)
print(total_return_orginal,total_return_optimized)
print(volatility_orginal,volatility_optimized)
print(total_return_optimized)

mdd_opt = max_drawdown(cr_2025)    # 优化组合的最大回撤
mdd_org = max_drawdown(ret_2025)  # 指数的最大回撤
print("最大回撤",mdd_opt,mdd_org)